# Keep your tool when you switch harnesses

Give two harnesses the same order lookup, prompt, and model. Only the harness setting changes.

Run the cells in order. You need an [OpenAI API key](https://platform.openai.com/api-keys) with API credit.

[Open in Colab](https://colab.research.google.com/github/BerriAI/liteagents/blob/main/cookbook/recipes/10_harness_switch.ipynb)

## 1. Install

Install LiteAgents and the integrations used in this notebook.

In [ ]:
%pip install -q --progress-bar off "liteagents[pydantic-ai,claude-sdk] @ https://github.com/BerriAI/liteagents/releases/download/v0.3.0a5/liteagents-0.3.0a5-py3-none-any.whl"

## 2. Add your key

Run this cell, paste your key into the hidden input, and press Enter.

In [ ]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = (os.environ.get("OPENAI_API_KEY") or getpass("OpenAI API key: ")).strip()
if not os.environ["OPENAI_API_KEY"]:
    raise ValueError("Run this cell again and enter your OpenAI API key.")

## 3. Define the shared tool

This is the same order lookup used in the [Python tools notebook](https://colab.research.google.com/github/BerriAI/liteagents/blob/main/cookbook/recipes/08_application_tools.ipynb).

In [ ]:
from typing import ClassVar

from liteagents import Tool


class LookupOrder(Tool):
    name = "lookup_order"
    description = "Look up an order's payment status and total."
    input_schema: ClassVar[dict] = {
        "type": "object",
        "properties": {"order_id": {"type": "string"}},
        "required": ["order_id"],
    }

    async def execute(self, input):
        print(f"Looking up {input['order_id']}")
        return f"Order {input['order_id']}: paid, total USD 12"

## 4. Run with Pydantic AI

You should see the tool look up A123 and the agent report **paid, USD 12**.

In [ ]:
from liteagents import ProfileOptions, run

profile = ProfileOptions(
    harness="pydantic-ai",
    model="openai/gpt-5.4-mini",
)
tools = [LookupOrder()]
prompt = "Use lookup_order for A123 and tell me its payment status and total."

first = await run(prompt, profile=profile, tools=tools)
print(first.text)

## 5. Change only the harness

The Claude harness uses the same OpenAI model and the same Python tool. Both integrations were installed in step 1.

In [ ]:
profile.harness = "claude-sdk"
second = await run(prompt, profile=profile, tools=tools)
print(second.text)

## 6. Inspect the tool calls

Both results use the application name `lookup_order`, so your event handling can stay the same too.

In [ ]:
from liteagents import AssistantMessage, ToolUseBlock

for result in [first, second]:
    for message in result.messages:
        if isinstance(message, AssistantMessage):
            for block in message.content:
                if isinstance(block, ToolUseBlock):
                    print(result.harness, block.name, block.input)

Try another installed harness by changing `profile.harness`. [Installation options](https://github.com/BerriAI/liteagents/blob/main/docs/getting-started.md#install-other-harnesses) cover DeepAgents, Codex, and OpenCode.

Explore [MCP tools](https://colab.research.google.com/github/BerriAI/liteagents/blob/main/cookbook/recipes/02_mcp.ipynb), [conversation history](https://colab.research.google.com/github/BerriAI/liteagents/blob/main/cookbook/recipes/01_quickstart.ipynb), and [Temporal recovery](https://colab.research.google.com/github/BerriAI/liteagents/blob/main/cookbook/recipes/06_durable.ipynb) in their own walkthroughs.

[Other model providers](https://github.com/BerriAI/liteagents/blob/main/docs/models.md) · [All cookbooks](https://github.com/BerriAI/liteagents/blob/main/cookbook/README.md)